# WCAG Conformance Estimater
This utility fetches the raw HTML of a page, sends it to the LLM, asks it to estimate the [WCAG conformance level](https://www.w3.org/WAI/WCAG22/Understanding/conformance) (A, AA, AAA) based on structural signals it can see in the markup — presence of lang attribute, alt text on images, label associations on form fields, heading hierarchy, landmark regions.


In [ ]:
# imports

import sys
sys.path.insert(0, '..')

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI


# Connecting to OpenAI

Load in the environment variables and connect to OpenAI. 
If you'd like to use free Ollama instead, please see the README section "Free Alternative to Paid APIs", and if you're not sure how to do this, there's a full solution in the solutions folder (day1_with_ollama.ipynb).

## Troubleshooting if you have problems:

If you get a "Name Error" - have you run all cells from the top down? Head over to the Python Foundations guide for a bulletproof way to find and fix all Name Errors.

If that doesn't fix it, head over to the [troubleshooting](../setup/troubleshooting.ipynb) notebook for step by step code to identify the root cause and fix it!

Or, contact me! Message me or email ed@edwarddonner.com and we will get this to work.

Any concerns about API costs? See my notes in the README - costs should be minimal, and you can control it at every point. You can also use Ollama as a free alternative, which we discuss during Day 2.

In [ ]:
# Load environment variables

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
openai = OpenAI()

In [ ]:
# Define our system prompt.

system_prompt = """
You are an accessibility auditor with expert knowledge of the WCAG 2.2 conformance criteria. You are given the raw HTML of a web page, and your job is to estimate the WCAG conformance level (A, AA, AAA) based on structural signals you see in the markup — presence of lang attribute, alt text on images, label associations on form fields, heading hierarchy, landmark regions.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [ ]:
# Define our user prompt

user_prompt_prefix = """
Here is the raw HTML of a website.
Estimate the WCAG conformance level for this website.
"""

In [ ]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [ ]:
# And now: call the OpenAI API.

def estimate_conformance(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [ ]:
# A function to display this nicely in the output, using markdown

def display_estimate(url):
    estimate = estimate_conformance(url)

    display(Markdown(estimate))

# Estimate the WCAG conformance level of a page with many accessibility violations.

In [ ]:
display_estimate("https://www.w3.org/WAI/demos/bad/before/home.html")